<a href="https://colab.research.google.com/github/RiazullJannat/ML/blob/main/ML_M_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.linear_model import LogisticRegression

**1. Load the dataset**


In [26]:
# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv("./sample_data/titanic_dataset.csv")

# Display 5 random rows to see what the data looks like
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
99,100,no,second,"Kantor, Mr. Sinai",male,34.0,1,0,244367,26.0000,NaN,S
496,497,yes,first,"Eustis, Miss. Elizabeth Mussey",female,54.0,1,0,36947,78.2667,D20,C
386,387,no,third,"Goodwin, Master. Sidney Leonard",male,1.0,5,2,CA 2144,46.9000,NaN,S
508,509,no,third,"Olsen, Mr. Henry Margido",male,28.0,0,0,C 4001,22.5250,NaN,S
618,619,yes,second,"Becker, Miss. Marion Louise",female,4.0,2,1,230136,39.0000,F4,S


In [27]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


**2. Feature Engineering**

In [28]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
736,737,no,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.00,1,3,W./C. 6608,34.3750,Missing,S,5,M
777,778,yes,third,"Emanuel, Miss. Virginia Ethel",female,5.00,0,0,364516,12.4750,Missing,S,1,M
391,392,yes,third,"Jansson, Mr. Carl Olof",male,21.00,0,0,350034,7.7958,Missing,S,1,M
190,191,yes,second,"Pinsky, Mrs. (Rosa)",female,32.00,0,0,234604,13.0000,Missing,S,1,M
644,645,yes,third,"Baclini, Miss. Eugenie",female,0.75,2,1,2666,19.2583,Missing,C,4,M


In [29]:
df['Deck'].value_counts()

,count
Deck,
M,687
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


**3. Data splitting**

In [30]:
X = df.drop(["Survived","SibSp","Parch", "Cabin","Name","PassengerId"], axis=1)
y = df["Survived"]


X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

**4. Out Lier Handiling and Data Cleaning**

In [31]:
#Age
mean_of_age = df['Age'].mean()
std_of_age = df['Age'].std()

X_train['age-z-score'] = (X_train['Age']-mean_of_age)/std_of_age
age_musk = (abs(X_train['age-z-score']) <=3)
X_train = X_train[age_musk]
y_train = y_train[age_musk]

#fare
fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

min_fare = max(0,fare_Q1 - 1.5 * IQR)
max_fare = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train["Fare"].clip(min_fare, max_fare)


**5. Building Preprocessing Pipelines**

In [32]:
#numerical pipelines

p1 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy='median')),
        ("scaler", MinMaxScaler())
    ]
)



#categorical pipelines

p3 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoading', OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)


p4 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoading', OrdinalEncoder(categories=[['third','second','first']],handle_unknown='ignore')),
        ('scaler', MinMaxScaler())
    ]
)

In [33]:
#transformer

preprocessor = ColumnTransformer(
    transformers=[
        ("pipeline_1", p1, ['Age']),
        ("pipeline_2", p2, ['Fare','Family_Size']),
        ("pipeline_3", p3, ['Embarked','Sex','Deck']),
        ("pipeline_4", p4, ['Pclass'])
    ],
    remainder="drop"
)

preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoading',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoading',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']],
                                                                 handle_unknown='ignore')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

In [37]:
#Label encoading

le = LabelEncoder()
le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)